# Sheet 4: ODEs and dynamical systems

Worked solutions, with each subpart labelled. Run the notebook from top to bottom. No data files are needed. The required packages are NumPy, SciPy and Matplotlib.

The saved figures are static. For interactive 3D rotation in Jupyter or VS Code notebooks, install ipympl, restart the kernel, then run `%matplotlib widget` before creating figures. For PyCharm or Spyder, use an external interactive plotting window as described in the question sheet. These optional display settings do not change the calculations.

The tables use `np.round(values, 6)` to display up to six decimal places. This changes only the displayed copy; the calculations keep the original values. Very small errors are printed without rounding so they remain visible.

References: the Class 6 lecture and try-it notebook; [SciPy solve_ivp documentation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.solve_ivp.html); [ACME predator-prey models](https://labs.acme.byu.edu/Volume4/PredatorPreyModels/PredatorPreyModels.html).

In [ ]:
# Import the numerical solver and plotting libraries.
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp


Display settings: `precision=6` limits printed arrays to six decimal places; `suppress=False` lets tiny errors appear in scientific notation. Neither changes the stored numbers. The plots use Matplotlib's default fonts and line widths.

In [ ]:
# Set readable numeric output.
np.set_printoptions(precision=6, suppress=False)


## Question 1: Linear dynamics

### (a) Matrix, equilibrium and eigenvalues

The matrix and equilibrium are

$$
A=\begin{pmatrix}-1&2\\0&-2\end{pmatrix},\qquad z^*=(0,0)^T.
$$

The eigenvalues are negative, so every trajectory of this linear system approaches the origin. A component need not decrease at every time: initially the input from y makes x increase.

In [ ]:
# Build the matrix, then inspect eigenvalues and initial rates.
A_linear = np.array([[-1., 2.], [0., -2.]])
initial_linear = np.array([1., 1.])
print("Eigenvalues:", np.linalg.eigvals(A_linear))
print("Initial derivatives [dx/dt, dy/dt]:", A_linear @ initial_linear)


### (b) Define the derivative function and solve

The function returns the two derivatives, not the next state. The solver supplies both t and state whenever it needs an evaluation. This autonomous model does not use t explicitly. Each column of `sol.y` is the state at the corresponding entry of `sol.t`.

Why these settings? The two decay rates are -1 and -2, so the interval 0 to 6 shows the approach to zero clearly. The initial state (1,1) makes both components active and reveals the brief initial rise of x. The 301 saved values give output spacing 0.02 for a smooth graph; they do not prescribe internal solver steps. The tolerances are chosen to make numerical discrepancies small enough for comparison with the exact solution.

The options `rtol` and `atol` control the estimated local error: relative tolerance scales with the state magnitude, while absolute tolerance supplies an allowance near zero. They do not specify a guaranteed bound on the accumulated error. These numerical controls are separate from colours, grids and other drawing choices.

In [ ]:
# Define the derivative function.
def linear_rhs(t, state):
    return A_linear @ state


In [ ]:
# Solve at the requested output times.
linear_times = np.linspace(0., 6., 301)
linear_sol = solve_ivp(linear_rhs, (0., 6.), initial_linear,
                       t_eval=linear_times, rtol=1e-6, atol=1e-9)
print("Success:", linear_sol.success)
print("Message:", linear_sol.message)
print("sol.t shape:", linear_sol.t.shape)
print("sol.y shape:", linear_sol.y.shape)
print("Initial state, sol.y[:, 0]:", linear_sol.y[:, 0])
print("Final state, sol.y[:, -1]:", linear_sol.y[:, -1])


### (c) Time and phase plots

The first coordinate increases briefly before decaying. The second decays immediately. The phase plot shows one path through state space, rather than a separate curve for each coordinate against time.

Plot the two coordinates against time first.

`fig` is the whole figure and `ax` is one plotting area. `ax.plot(horizontal, vertical)` draws a curve; `label` supplies its legend entry. Matplotlib gives successive curves different colours automatically. `ax.set` assigns axis labels and a title; it does not change the data. `ax.legend()` identifies the curves and `plt.show()` displays the figure.

In [ ]:
# Plot the two state coordinates against time.
fig, ax = plt.subplots()
ax.plot(linear_sol.t, linear_sol.y[0], label="x(t)")
ax.plot(linear_sol.t, linear_sol.y[1], label="y(t)")
ax.set(xlabel="Time t", ylabel="State", title="Both coordinates approach zero")
ax.legend()  # Identify curves using their label strings.
plt.show()


Now plot x against y, so that each plotted point represents one complete state.

The format `"o"` draws circular markers without joining them. We use orange for the initial state and black for the equilibrium so their roles are easy to recognise. These colours carry no mathematical meaning. The point markers are drawn after the trajectory using ordinary `plot` calls.

In [ ]:
# Draw the phase trajectory and mark its start and equilibrium.
fig, ax = plt.subplots()
ax.plot(linear_sol.y[0], linear_sol.y[1], label="Trajectory")
ax.plot([1.], [1.], "o", color="orange", label="Start")
ax.plot([0.], [0.], "o", color="black", label="Equilibrium")
ax.set(xlabel="x", ylabel="y", title="Phase trajectory")
ax.legend()
plt.show()


### (d) Exact solution and requested times

At time zero the given formulas give (1,1). Their derivatives satisfy

$$
\dot y=-2e^{-2t}=-2y,\qquad
\dot x=-3e^{-t}+4e^{-2t}=-x+2y.
$$

We integrate from zero again, with the same initial condition, but save only the three requested times. The first stored column now corresponds to time 0.5, not time zero.

In [ ]:
# Define the exact state as a function of time.
def exact_linear(t):
    t = np.asarray(t)
    return np.array([3*np.exp(-t)-2*np.exp(-2*t), np.exp(-2*t)])


In [ ]:
# Compare exact and computed states at three selected times.
requested_linear = np.array([0.5, 2., 5.])
linear_at_times = solve_ivp(linear_rhs, (0., 6.), initial_linear,
                           t_eval=requested_linear, rtol=1e-6, atol=1e-9)
exact_requested = exact_linear(requested_linear)
print("Columns: t, computed x, computed y, exact x, exact y")
print(np.round(np.column_stack([requested_linear, linear_at_times.y.T,
                                exact_requested.T]), 6))
print("Absolute errors: rows x and y, columns requested times")
print(np.abs(linear_at_times.y-exact_requested))


### (e) Dense output

`linear_dense.sol` is an interpolator built by the solver. We give it times and it evaluates the computed trajectory there. Indexing `linear_dense.y` instead selects values already stored in its columns. Dense output is an approximation to the solution, not a new exact formula.

In [ ]:
# Evaluate dense output and compare it with the exact state.
linear_dense = solve_ivp(linear_rhs, (0., 6.), initial_linear,
                         t_eval=linear_times, dense_output=True,
                         rtol=1e-6, atol=1e-9)
dense_times = np.array([0.37, 1.23, 4.56])
dense_states = linear_dense.sol(dense_times)
print("Columns: t, interpolated x, interpolated y")
print(np.round(np.column_stack([dense_times, dense_states.T]), 6))
print("Absolute errors:")
print(np.abs(dense_states-exact_linear(dense_times)))


## Question 2: Nonlinear equilibria

### (a) Jacobian and local stability

At equilibrium, $v=0$ and $x-x^3=0$. Thus the equilibria are (-1,0), (0,0), (1,0). The Jacobian is

$$
J(x,v)=\begin{pmatrix}0&1\\1-3x^2&-0.4\end{pmatrix}.
$$

At the two outer equilibria the eigenvalues have negative real parts: both are locally asymptotically stable. At the origin one eigenvalue is positive: it is an unstable saddle. These tests concern nearby initial states; they do not by themselves determine the destination of every trajectory.

In [ ]:
# Define the nonlinear derivatives and Jacobian.
def motion(t, state):
    x, velocity = state
    return [velocity, x-x**3-0.4*velocity]


In [ ]:
# Define the analytic Jacobian for any position.
def motion_jacobian(x):
    return np.array([[0., 1.], [1.-3*x*x, -0.4]])


In [ ]:
# Evaluate the Jacobian at each equilibrium.
motion_equilibria = np.array([[-1., 0.], [0., 0.], [1., 0.]])
for equilibrium in motion_equilibria:
    print("Equilibrium:", equilibrium,
          "Eigenvalues:", np.linalg.eigvals(motion_jacobian(equilibrium[0])))


### (b) Compare initial states

We expect the small positive displacement to move toward the positive equilibrium, and the small negative displacement toward the negative equilibrium. The state (1.1,0) is already near the positive equilibrium. The simulations confirm these predictions for the three chosen starts.

First compute the three trajectories. Store the solver results so we can reuse them in later plots and tables.

The starts (0.05,0) and (-0.05,0) lie just to opposite sides of the unstable equilibrium; (1.1,0) lies near the positive stable equilibrium. Damping 0.4 allows visible oscillations that gradually decay. Integrating to 40 lets us see the later settling. Saving 2001 points gives spacing 0.02; the tighter tolerances help distinguish numerical error from the different destinations.

In [ ]:
# Solve once for each initial state.
motion_initials = [[0.05, 0.], [-0.05, 0.], [1.1, 0.]]
motion_times = np.linspace(0., 40., 2001)
motion_solutions = []
for initial in motion_initials:
    result = solve_ivp(motion, (0., 40.), initial, t_eval=motion_times,
                       dense_output=True, rtol=1e-7, atol=1e-9)
    motion_solutions.append(result)


Check each run before interpreting it.

In [ ]:
# Report whether the solve succeeded and where it ended.
for initial, result in zip(motion_initials, motion_solutions):
    print("Start:", initial, "Success:", result.success,
          "Final state:", np.round(result.y[:, -1], 6))


Compare the positions against time.

In [ ]:
# Plot the position coordinate from each solution.
fig, ax = plt.subplots()
for initial, result in zip(motion_initials, motion_solutions):
    ax.plot(result.t, result.y[0], label=str(initial))
ax.set(xlabel="Time t", ylabel="Position x", title="Different starting positions")
ax.legend()
plt.show()


Compare the paths in position-velocity space.

In [ ]:
# Plot each complete state trajectory and the three equilibria.
fig, ax = plt.subplots()
for initial, result in zip(motion_initials, motion_solutions):
    ax.plot(result.y[0], result.y[1], label=str(initial))
ax.plot(motion_equilibria[:, 0], motion_equilibria[:, 1], "o",
           color="black", label="Equilibria")
ax.set(xlabel="Position x", ylabel="Velocity v", title="Phase trajectories")
ax.legend()
plt.show()


### (c) States at selected times

The two starts close to the origin end near different equilibria. Being close initially does not imply the same eventual destination in a system with several attracting equilibria.

In [ ]:
# Evaluate and print the nonlinear states at four times.
motion_query_times = np.array([5., 10., 20., 40.])
for initial, result in zip(motion_initials[:2], motion_solutions[:2]):
    print("Initial state:", initial)
    print("Columns: time, x, velocity")
    print(np.round(np.column_stack([motion_query_times,
                                    result.sol(motion_query_times).T]), 6))
    print("Final distances to (-1,0), (0,0), (1,0):")
    print(np.linalg.norm(motion_equilibria-result.y[:, -1], axis=1))


### (d) Vector field

`meshgrid` repeats the position and velocity coordinates to build a rectangular grid. At each grid point we compute the two derivatives. The arrows point along the trajectory tangents; their drawn endpoints are not completed numerical integration steps. We shorten them for readability.

The chosen coordinate limits include the three equilibria and the trajectories. A 19 by 17 grid gives enough arrows to show the flow without crowding the figure. `angles="xy"` gives directions in the data coordinates; `scale_units="xy"` uses those coordinate units for length. `scale=9` shortens arrows by a factor of 9. `color="0.65"` is a light grey (0 is black, 1 is white), keeping the coloured trajectories easy to see. None of these settings changes the trajectories.

In [ ]:
# Build the state grid and calculate the derivatives.
positions = np.linspace(-1.6, 1.6, 19)
velocities = np.linspace(-1.2, 1.2, 17)
X_motion, V_motion = np.meshgrid(positions, velocities)
dX_motion = V_motion
dV_motion = X_motion-X_motion**3-0.4*V_motion


Overlay the saved trajectories on the derivative arrows.

In [ ]:
# Draw the field and the previously computed trajectories.
fig, ax = plt.subplots()
ax.quiver(X_motion, V_motion, dX_motion, dV_motion,
          angles="xy", scale_units="xy", scale=9, color="0.65")
for initial, result in zip(motion_initials, motion_solutions):
    ax.plot(result.y[0], result.y[1], label=str(initial))
ax.plot(motion_equilibria[:, 0], motion_equilibria[:, 1], "o", color="black")
ax.set(xlim=(-1.6, 1.6), ylim=(-1.2, 1.2), xlabel="Position x", ylabel="Velocity v",
       title="Derivatives and trajectories")
ax.legend()
plt.show()


## Question 3: Steps and accuracy

### (a) Euler updates

Substituting the derivative gives

$$
u_{n+1}=u_n-2hu_n=(1-2h)u_n.
$$

For h=0.25 the multiplier is 0.5: positive decay. For h=0.75 it is -0.5: alternating decay. For h=1.25 it is -1.5: alternating growth. The latter two introduce signs absent from the exact positive solution.

The three step sizes deliberately show positive decay, alternating decay, and growth beyond the stability limit. All divide 15 exactly, so their numerical runs stop at the same final time.

In [ ]:
# Calculate three Euler updates for each step size.
euler_steps = [0.25, 0.75, 1.25]
for h in euler_steps:
    value = 1.
    first_three = []
    for step in range(3):
        value = value+h*(-2*value)
        first_three.append(value)
    print("h:", h, "Multiplier:", 1-2*h, "First three updates:", first_three)


### (b) Implement Euler and compare

Decay of numerical deviations requires |1-2h|<1, which for positive h gives

$$0<h<1.$$

At h=1 the multiplier is -1, giving persistent alternation rather than decay. The stability restriction is different from a requirement for small approximation error.

In [ ]:
# Implement the Euler updates.
def euler_decay(h):
    n_steps = int(round(15/h))
    times = h*np.arange(n_steps+1)
    values = np.empty(n_steps+1)
    values[0] = 1.
    for step in range(n_steps):
        values[step+1] = values[step]+h*(-2*values[step])
    return times, values


Plot style: `plt.subplots(1,3)` creates one row with three plotting areas, one for each step size. `figsize=(12,3.5)` gives them enough horizontal space. The format `"o-"` draws the Euler values as circles joined by lines; `markersize=3` keeps the circles small. `"k--"` draws the exact curve as a black dashed line, so it remains distinguishable from the numerical curve. The three panels have separate vertical scales because the unstable approximation grows much larger.

In [ ]:
# Plot each step size against the exact solution.
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
exact_plot_times = np.linspace(0., 15., 601)
for ax, h in zip(axes, euler_steps):
    times, values = euler_decay(h)
    ax.plot(times, values, "o-", markersize=3, label="Euler")
    ax.plot(exact_plot_times, np.exp(-2*exact_plot_times), "k--", label="Exact")
    ax.set(xlabel="Time t", ylabel="u", title="h = " + str(h))
    ax.legend()
plt.show()


### (c) RK45 saved points

Requesting output every unit of time does not fix RK45's internal step size. The method selects its own steps and interpolates to the requested output times.

In [ ]:
# Define the scalar derivative.
def scalar_decay(t, state):
    return -2*state


In [ ]:
# Solve with sparsely saved output.
rk_sparse = solve_ivp(scalar_decay, (0., 15.), [1.], method="RK45",
                       t_eval=np.arange(0., 16.))
print("Success:", rk_sparse.success, "Saved times:", len(rk_sparse.t),
      "Derivative evaluations:", rk_sparse.nfev)
fig, ax = plt.subplots()
ax.plot(exact_plot_times, np.exp(-2*exact_plot_times), "k-", label="Exact")
ax.plot(rk_sparse.t, rk_sparse.y[0], "o", label="RK45 saved values")
ax.set(xlabel="Time t", ylabel="u", title="Saved values do not specify internal steps")
ax.legend()
plt.show()


### (d) Tolerances and accuracy

Both runs use the same output grid. Tighter tolerances usually require more derivative evaluations and reduce the error here. They control estimates of local error, not a guaranteed upper bound on the final accumulated error. The output grid determines where we inspect the solution.

In [ ]:
# Compare two tolerance choices on the same output grid.
decay_output_times = np.linspace(0., 15., 301)
decay_comparison = []
for relative, absolute in [(1e-3, 1e-6), (1e-8, 1e-11)]:
    result = solve_ivp(scalar_decay, (0., 15.), [1.],
                       t_eval=decay_output_times, rtol=relative, atol=absolute)
    max_error = np.max(np.abs(result.y[0]-np.exp(-2*result.t)))
    decay_comparison.append((result.nfev, max_error))
    print("rtol:", relative, "atol:", absolute,
          "nfev:", result.nfev, "Maximum absolute error:", max_error)


### (e) A stiff system: RK45 and Radau

The component v decays over a time scale of about 0.001, whereas u changes over a time scale of about 1. The exact solution is

$$u(t)=e^{-t},\qquad v(t)=e^{-1000t}.$$

We use the same tolerances and output grid for both methods. The close-up makes the early rapid decay visible. Their curves mostly overlap.

Define the derivatives and construct the shared output grid.

The coefficient 1000 makes v decay 1000 times faster than u. Starting both at 1 exposes the fast transient as well as the slower motion. The first grid saves every 0.0001 time units until 0.02, so the rapid decay is visible. The later grid saves about every 0.02 units until 10, which is sufficient to display the slow component. Slicing with `[1:]` removes the repeated joining time. Using the same grid and tolerances for both methods keeps the comparison consistent.

In [ ]:
# Return the slow and fast decay rates in the same order as the state.
def stiff_decay(t, state):
    u, v = state
    return [-u, -1000*v]

# Save more points during the rapid initial decay.
early_times = np.linspace(0., 0.02, 201)
later_times = np.linspace(0.02, 10., 501)[1:]
stiff_times = np.concatenate([early_times, later_times])
stiff_exact = np.array([np.exp(-stiff_times), np.exp(-1000*stiff_times)])


Run the same problem with each solver.

In [ ]:
# Keep the tolerances and requested times identical.
stiff_results = {}
for method in ["RK45", "Radau"]:
    result = solve_ivp(stiff_decay, (0., 10.), [1., 1.], method=method,
                       t_eval=stiff_times, rtol=1e-6, atol=1e-9)
    stiff_results[method] = result


Measure elapsed time separately from the derivative count. The previous solves have warmed up both methods. Repeat each solve five times and report the median time to reduce the effect of timing fluctuations. The timer covers only the solve; preparing the grid, printing and plotting happen outside it. Run this cell on your own machine to obtain your timings.

In [ ]:
# Time repeated solves using the same problem, tolerances and saved times.
from time import perf_counter

stiff_timings = {"RK45": [], "Radau": []}
for repeat in range(5):
    for method in ["RK45", "Radau"]:
        start = perf_counter()
        timed_result = solve_ivp(stiff_decay, (0., 10.), [1., 1.], method=method,
                                 t_eval=stiff_times, rtol=1e-6, atol=1e-9)
        elapsed = perf_counter() - start
        stiff_timings[method].append(elapsed)


Compare the median measured times alongside the derivative evaluations and errors. `perf_counter()` measures elapsed time in seconds; `result.nfev` counts derivative evaluations. They are different quantities.

In [ ]:
# Compute a separate maximum error for each state coordinate.
for method, result in stiff_results.items():
    errors = np.max(np.abs(result.y-stiff_exact), axis=1)
    print("Method:", method, "Success:", result.success)
    print("Derivative evaluations:", result.nfev)
    print("Median solve time (seconds, 5 runs):", np.median(stiff_timings[method]))
    print("Maximum errors [u,v]:", errors)


Plot slow component u over the full interval.

In [ ]:
# Overlay the numerical and exact solutions.
fig, ax = plt.subplots()
for method, result in stiff_results.items():
    ax.plot(result.t, result.y[0], label=method)
ax.plot(stiff_times, stiff_exact[0], "k--", label="Exact")
ax.set(xlabel="Time t", ylabel="u", title="Slow component u")
ax.legend()
plt.show()


Plot fast component v over the full interval.

In [ ]:
# Overlay the numerical and exact solutions.
fig, ax = plt.subplots()
for method, result in stiff_results.items():
    ax.plot(result.t, result.y[1], label=method)
ax.plot(stiff_times, stiff_exact[1], "k--", label="Exact")
ax.set(xlabel="Time t", ylabel="v", title="Fast component v")
ax.legend()
plt.show()


Zoom in on the initial fast decay using a mask on the saved times.

In [ ]:
# Inspect the first 0.02 time units without solving again.
early = stiff_times <= 0.02
fig, ax = plt.subplots()
for method, result in stiff_results.items():
    ax.plot(result.t[early], result.y[1, early], label=method)
ax.plot(stiff_times[early], stiff_exact[1, early], "k--", label="Exact")
ax.set(xlabel="Time t", ylabel="v", title="Fast decay: close-up")
ax.legend()
plt.show()


Radau uses far fewer derivative evaluations in this example. As discussed in the lecture, a rapidly decaying component can impose a small stability limit on an explicit method even after its value is tiny. RK45 therefore continues taking many small steps. Radau is implicit and handles this stiffness more efficiently here. Derivative evaluations describe only part of the work: implicit methods also solve equations internally. Compare the measured times above as well: fewer derivative evaluations do not by themselves guarantee a shorter runtime. The times depend on the machine, software versions and other work running on the computer. Similar solution curves can therefore accompany different computational costs.

## Question 4: Predator and prey

### Understanding the model

Predator-prey models describe how two populations affect each other's growth. Let $R$ and $P$ denote the prey and predator populations. We work with scaled variables

$$
x=\frac{R}{R_0},\qquad y=\frac{P}{P_0},\qquad t=\frac{T}{T_0},
$$

where $R_0,P_0$ are reference population sizes and $T_0$ is a reference time. Thus $x=1.5$ means 1.5 times the reference prey population, not 1.5 animals. Scaling changes the units used to describe the system. We choose units that make the prey growth and interaction coefficients equal to 1. The remaining parameters $a$ and $d$ are dimensionless. This is an illustrative model; no particular real population units are specified.

The scaled model is

$$
\begin{aligned}
\dot x &= x-a x^2-xy=x(1-a x-y),\\
\dot y &= xy-dy=y(x-d),
\end{aligned}
\qquad a\ge0,\quad d>0.
$$

The term $+x$ describes prey growth. The term $-a x^2$ reduces growth through competition for resources. The term $-xy$ describes prey being eaten; the corresponding $+xy$ supports predator growth. Finally, $-dy$ describes predator deaths. A larger $d$ means faster predator loss when prey are scarce.

Carrying capacity. Without predators ($y=0$), the prey equation becomes

$$
\dot x=x(1-a x)=x\left(1-\frac{x}{K}\right),\qquad K=\frac{1}{a}\quad(a>0).
$$

Below $K$ the prey grow; above $K$ they decline. The carrying capacity is the population level approached without predators, not a hard ceiling that can never be exceeded. In original population units it is $R_0K$. If $a=0$, this model has no carrying capacity: prey without predators grow exponentially.

Equilibria. At a fixed state both populations stop changing, so

$$
x^*(1-a x^*-y^*)=0,\qquad y^*(x^*-d)=0.
$$

We distinguish total extinction, $(0,0)$; a prey-only equilibrium, $(x^*,0)$ with $x^*>0$; and a coexistence equilibrium, $(x^*,y^*)$ with both coordinates positive. At coexistence both populations are constant. This differs from an orbit on which both species survive but their populations oscillate. We now calculate these coordinates and analyse their stability, using $d=1$ in parts (a)-(c).

### (a) Equilibria and Jacobian

For a=0 and d=1 the equilibrium equations are

$$x(1-y)=0,\qquad y(x-1)=0.$$

If x=0, the second equation forces y=0. If x is positive, the first equation forces y=1, and then the second forces x=1. Thus there are exactly two nonnegative equilibria: total extinction (0,0), and coexistence (1,1).

For the full parameterised model,

$$
J(x,y)=\begin{pmatrix}1-2ax-y&-x\\y&x-d\end{pmatrix}.
$$

For a=0, d=1 the equilibria are (0,0) and (1,1). At the origin the eigenvalues are 1 and -1, giving a saddle. At (1,1) they are $\pm$i. The nonlinear Jacobian test is inconclusive when the real parts are zero: imaginary eigenvalues alone do not prove periodic motion.

In [ ]:
# Define the population derivatives and their Jacobian.
def predator_prey(t, state, a, d):
    x, y = state
    return [x*(1-a*x)-x*y, x*y-d*y]


In [ ]:
# Define the analytic Jacobian for a population state.
def predator_jacobian(x, y, a, d):
    return np.array([[1-2*a*x-y, -x], [y, x-d]])


In [ ]:
# Check the two classic equilibria.
for point in [[0., 0.], [1., 1.]]:
    print("Equilibrium:", point, "Eigenvalues:",
          np.linalg.eigvals(predator_jacobian(point[0], point[1], 0., 1.)))


### (b) Population cycles

In these trajectories the prey peaks precede the predator peaks: growing prey supports predator growth, which then suppresses prey. The requested start generates a persistent cycle around (1,1), rather than spiralling into it.

In [ ]:
# Compute the classic predator-prey trajectory.
prey_short_times = np.linspace(0., 30., 1501)
prey_classic = solve_ivp(predator_prey, (0., 30.), [1.5, 0.5],
                         args=(0., 1.), t_eval=prey_short_times,
                         rtol=1e-8, atol=1e-10)


Draw the time plot and the phase trajectory from the saved solution.

Here `plt.subplots(1,2)` creates two panels: the first uses time on its horizontal axis; the second uses prey population. `figsize=(10,4.2)` gives both panels space. Orange circles mark the initial state and black circles the equilibrium.

The legends are above the panels so they cannot cover the equilibrium. `loc="lower center"` anchors the bottom centre of each legend at `bbox_to_anchor=(0.5,1.02)`: halfway across the plotting area and slightly above it. These are relative plot coordinates, not population values. `ncol=3` places entries in one horizontal row, `fontsize=9` keeps them compact, and title `pad=35` leaves room above them. These settings only arrange the labels.

In [ ]:
# Draw two complementary views of the same solution.
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
axes[0].plot(prey_classic.t, prey_classic.y[0], label="Prey x")
axes[0].plot(prey_classic.t, prey_classic.y[1], label="Predators y")
axes[0].set(xlabel="Time t", ylabel="Population")
axes[0].set_title("Population cycles", pad=35)
axes[1].plot(prey_classic.y[0], prey_classic.y[1], label="Trajectory")
axes[1].plot([1.5], [0.5], "o", color="orange", label="Start")
axes[1].plot([1.], [1.], "o", color="black", label="Equilibrium")
axes[1].set(xlabel="Prey x", ylabel="Predators y")
axes[1].set_title("Persistent orbit", pad=35)
for ax in axes:
    # Keep the legend above the axes, away from the equilibrium.
    ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.02), ncol=3, fontsize=9)
plt.show()


For this particular classic model, the quantity x-log(x)+y-log(y) stays constant along positive trajectories. Its closed level curves around (1,1) explain the cycles. This is additional information about the nonlinear equations; it does not follow just from the imaginary eigenvalues.

### (c) How parameters change the equilibria

A coexistence equilibrium means both species are present with constant, positive populations. Both derivatives are zero there; it is not an oscillation in which both species happen to survive. A prey-only equilibrium instead has zero predators.

To find them, set both factored derivatives equal to zero. With x=0 the second equation gives y=0. With y=0 and x positive, the first equation gives 1-ax=0, hence x=1/a. With both populations positive, we can divide out x and y, obtaining

$$1-ax-y=0,\qquad x-d=0.$$

Therefore x=d and y=1-ad. For a>0 the prey-only and coexistence candidates are:

$$
E_P=(1/a,0),\qquad E_C=(d,1-ad).
$$

Coexistence requires ad<1. At the prey-only equilibrium the eigenvalues are -1 and 1/a-d: it is locally asymptotically stable for ad>1 and a saddle for ad<1. At coexistence,

$$
J(E_C)=\begin{pmatrix}-ad&-d\\1-ad&0\end{pmatrix},\qquad
\operatorname{tr}J=-ad,\quad \det J=d(1-ad).
$$

For a>0 and ad<1, both eigenvalues have negative real parts, so coexistence is locally asymptotically stable. With d=1 the range is 0<a<1. At ad=1 the equilibria meet and one eigenvalue is zero, so this linearization test is inconclusive.

In [ ]:
# Evaluate the equilibrium Jacobians for three a values.
for a in [0.25, 0.75, 1.25]:
    d = 1.
    prey_only = [1/a, 0.]
    print("a:", a, "Prey-only equilibrium:", prey_only,
          "Eigenvalues:", np.linalg.eigvals(predator_jacobian(prey_only[0], prey_only[1], a, d)))
    if a*d < 1:
        coexist = [d, 1-a*d]
        print("Coexistence:", coexist, "Eigenvalues:",
              np.linalg.eigvals(predator_jacobian(coexist[0], coexist[1], a, d)))


### (d) Six parameter pairs and vector fields

Use rows for a and columns for d. On each panel we calculate the derivatives on a grid, draw the arrows and overlay one simulated trajectory. All plots include only the nonnegative population region. The origin is also an equilibrium.

For these examples: a=0 gives persistent cycles for both d values; a=0.25 gives convergence to coexistence for both; a=1.25 gives coexistence for d=0.5 but predator loss for d=2. At coexistence, increasing d moves x* to the right and decreases y*=1-ad. The prey-only equilibrium depends on a, not d.

Why these parameter values? With a=0 there is no limit on prey growth, giving the classic cycles. For a=0.25, both d choices satisfy ad<1, so coexistence is possible and locally stable. With a=1.25, d=0.5 still gives ad<1, whereas d=2 gives ad>1 and predator extinction. The grid therefore shows different behaviours with only six solves. The common initial state makes the effects of parameter changes easier to compare. The long interval up to 300 allows slowly settling trajectories to approach their equilibria.

First compute the six trajectories. Dictionary keys are the parameter pairs (a,d).

In [ ]:
# Solve for each combination of the two parameters.
a_grid = [0., 0.25, 1.25]
d_grid = [0.5, 2.]
prey_long_times = np.linspace(0., 300., 6001)
prey_grid_runs = {}
for a in a_grid:
    for d in d_grid:
        result = solve_ivp(predator_prey, (0., 300.), [1.5, 0.5], args=(a, d),
                           t_eval=prey_long_times, rtol=1e-8, atol=1e-10)
        prey_grid_runs[(a, d)] = result


In [ ]:
# Inspect each final state before drawing the phase plots.
for pair, result in prey_grid_runs.items():
    print("a, d:", pair, "Success:", result.success,
          "Final state:", np.round(result.y[:, -1], 6))


Use the same population grid in every panel. It includes all six trajectories and their nonnegative equilibria. The arrows show population derivatives, not a gradient flow; all panels use the same arrow scaling.

The ranges 0 to 4.5 for prey and 0 to 3 for predators include the relevant fixed points and all six paths. We use 19 by 17 coordinate positions as a readable sampling of the field, not as integration steps. Passing both coordinate matrices into `predator_prey` evaluates the formulas element by element and returns two derivative matrices of the same shape. The time argument 0 is unused in this autonomous model.

In [ ]:
# Build the coordinate arrays once; only the derivatives vary between panels.
prey_positions = np.linspace(0., 4.5, 19)
predator_positions = np.linspace(0., 3., 17)
X_prey, Y_prey = np.meshgrid(prey_positions, predator_positions)


A short helper draws one panel. Keeping it separate lets us read the plotting loop without mixing it with the numerical integration.

It receives an existing plotting area `ax`, the parameters and one stored solution. `scale=60` shortens arrows to avoid overlap; `color="0.6"` makes them grey so the coloured trajectory remains easy to see. Orange circles mark the common start (`markersize=5`); black crosses mark equilibria. The same `xlim` and `ylim` in every panel let us compare the parameter choices fairly. The arrow shaft and line widths use Matplotlib's defaults.

In [ ]:
# Draw one vector field, one trajectory, and the nonnegative fixed points.
def draw_prey_panel(ax, a, d, result):
    U, V = predator_prey(0., [X_prey, Y_prey], a, d)
    ax.quiver(X_prey, Y_prey, U, V, angles="xy", scale_units="xy",
              scale=60, color="0.6")
    ax.plot(result.y[0], result.y[1])
    ax.plot([1.5], [0.5], "o", color="orange", markersize=5)
    ax.plot([0.], [0.], "x", color="black")
    if a > 0:
        ax.plot([1/a], [0.], "x", color="black")
    if a*d <= 1:
        ax.plot([d], [1-a*d], "x", color="black")
    ax.set(xlim=(0., 4.5), ylim=(0., 3.),
           title="a=" + str(a) + ", d=" + str(d))


Finally arrange the panels: rows vary a, columns vary d. `plt.subplots(3,2)` makes three rows and two columns. `axes[row,column]` selects one panel; `enumerate` supplies the indices alongside the parameter values. The 11 by 11 inch figure leaves space for six panels. This loop repeats the same plotting instructions, keeping the presentation consistent. We label only the bottom row and left column, so labels do not crowd the spaces between panels. Each panel still has its own numeric tick labels.

In [ ]:
# Reuse the same plotting helper for the six parameter pairs.
fig, axes = plt.subplots(3, 2, figsize=(11, 11))
for row, a in enumerate(a_grid):
    for column, d in enumerate(d_grid):
        draw_prey_panel(axes[row, column], a, d, prey_grid_runs[(a, d)])
        if row == 2:
            axes[row, column].set_xlabel("Prey x")
        if column == 0:
            axes[row, column].set_ylabel("Predators y")
plt.show()


## Question 5: The Lorenz attractor

### (a) Define and solve the 3D system

The callback returns three derivatives in the same order as the three state coordinates. The result has three rows and 8001 saved columns. All runs below start at time zero and use the same requested output times.

The values sigma=10, rho=28 and beta=8/3 are the conventional Lorenz example used in the question. Starting at (1,1,1) gives a non-equilibrium trajectory. The interval 0 to 40 shows the early transient and later motion; 8001 saved times give spacing 0.005 for a smooth drawing. The tight tolerances reduce early numerical errors, but cannot guarantee long-term agreement in this sensitive system.

In [ ]:
# Define the three Lorenz derivatives.
def lorenz(t, state):
    x, y, z = state
    return [10*(y-x), x*(28-z)-y, x*y-(8./3)*z]


In [ ]:
# Integrate and inspect the returned arrays.
lorenz_times = np.linspace(0., 40., 8001)
lorenz_base = solve_ivp(lorenz, (0., 40.), [1., 1., 1.],
                        t_eval=lorenz_times, rtol=1e-9, atol=1e-12)
print("Success:", lorenz_base.success)
print("Message:", lorenz_base.message)
print("Time shape:", lorenz_base.t.shape, "State shape:", lorenz_base.y.shape)


### (b) Time series, 3D trajectory and a projection

The 3D plot shows all three coordinates together. The x-z projection discards y: two parts of the trajectory can appear to intersect in that projection even when separated in 3D.

`projection="3d"` creates a three-dimensional plotting area; its `plot` method receives x, y and z in that order. We use the default line width. In an interactive window, rotate the figure to inspect how the curves are arranged.

In [ ]:
# Plot all three coordinates against time.
fig, ax = plt.subplots()
for row, name in enumerate(["x", "y", "z"]):
    ax.plot(lorenz_base.t, lorenz_base.y[row], label=name)
ax.set(xlabel="Time t", ylabel="Coordinate", title="Lorenz coordinates")
ax.legend()
plt.show()


In [ ]:
# Draw the trajectory in 3D.
fig = plt.figure()
ax = fig.add_subplot(111, projection="3d")
ax.plot(lorenz_base.y[0], lorenz_base.y[1], lorenz_base.y[2])
ax.set(xlabel="x", ylabel="y", zlabel="z", title="Lorenz trajectory in 3D")
plt.show()


In [ ]:
# Draw the x-z projection.
fig, ax = plt.subplots()
ax.plot(lorenz_base.y[0], lorenz_base.y[2])
ax.set(xlabel="x", ylabel="z", title="Projection onto the x-z plane")
plt.show()


### (c) Sensitivity to initial conditions

Change just the first coordinate by one millionth. We compare both x(t) curves on a short and a long interval, then plot the Euclidean distance between the full states. The distance is initially $10^{-6}$. Its growth is not necessarily monotonic: separation can grow and shrink as the trajectories move through state space.

The perturbation 0.000001 is tiny compared with the initial coordinates of size 1. We use 0 to 5 to inspect early agreement and 0 to 40 to see later separation. A dashed line (`"--"`) distinguishes the perturbed trajectory when the two curves overlap. Small legend text (`fontsize=8`) helps it fit inside each comparison panel. The distance uses all three coordinates, while the overlaid time curves show only x.

In [ ]:
# Solve from the slightly changed initial state and compute distances.
lorenz_nearby = solve_ivp(lorenz, (0., 40.), [1.+1e-6, 1., 1.],
                          t_eval=lorenz_times, rtol=1e-9, atol=1e-12)
lorenz_distance = np.linalg.norm(lorenz_base.y-lorenz_nearby.y, axis=0)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, end in zip(axes, [5., 40.]):
    mask = lorenz_times <= end
    ax.plot(lorenz_times[mask], lorenz_base.y[0, mask], label="Original start")
    ax.plot(lorenz_times[mask], lorenz_nearby.y[0, mask], "--", label="Nearby start")
    ax.set(xlabel="Time t", ylabel="x(t)", title="0 to " + str(end))
    ax.legend(fontsize=8)
plt.show()


In [ ]:
# Plot the separation against time.
fig, ax = plt.subplots()
ax.plot(lorenz_times, lorenz_distance)
ax.set(xlabel="Time t", ylabel="Distance between states",
       title="Sensitivity to the initial state")
plt.show()
print("Initial distance:", lorenz_distance[0])
for t in [5., 10., 20., 40.]:
    index = np.argmin(np.abs(lorenz_times-t))
    print("Time:", t, "Distance:", lorenz_distance[index])


The curves initially lie almost on top of one another. Later they become visibly different although the initial change was tiny. This is an introduction to sensitivity to initial conditions: deterministic equations do not imply that nearby initial states remain close indefinitely. The broad shape of the trajectory can look similar even after the individual time histories separate.

### (d) Sensitivity to numerical error

Now the initial state stays exactly the same, but tighter tolerances change the small numerical errors introduced during integration. Compare this difference with the previous initial-state perturbation. A tighter run is a numerical comparison, not an exact reference trajectory.

In [ ]:
# Solve with tighter tolerances and compare the two sources of differences.
lorenz_tight = solve_ivp(lorenz, (0., 40.), [1., 1., 1.],
                         t_eval=lorenz_times, rtol=1e-11, atol=1e-13)
lorenz_accuracy_distance = np.linalg.norm(lorenz_base.y-lorenz_tight.y, axis=0)


Show both distances on the same axes for the early and full intervals.

In [ ]:
# Compare distances caused by the two small changes.
fig, axes = plt.subplots(1, 2, figsize=(11, 3.7))
for ax, end in zip(axes, [5., 40.]):
    mask = lorenz_times <= end
    ax.plot(lorenz_times[mask], lorenz_distance[mask], label="Changed initial state")
    ax.plot(lorenz_times[mask], lorenz_accuracy_distance[mask], label="Changed tolerances")
    ax.set(xlabel="Time t", ylabel="Distance between states", title="0 to " + str(end))
    ax.legend(fontsize=8)
plt.show()


Summarise the maximum distances on the two intervals.

In [ ]:
# Report maximum distances over each time interval.
for end in [5., 40.]:
    mask = lorenz_times <= end
    print("Interval end:", end,
          "Largest initial-state difference:", np.max(lorenz_distance[mask]),
          "Largest tolerance difference:", np.max(lorenz_accuracy_distance[mask]))


Small numerical differences can also be amplified. Early-time agreement can be excellent while long-time pointwise agreement is lost. Predicting one precise trajectory far ahead therefore needs very accurate initial data as well as careful numerical integration. The times and amounts of separation can vary with software versions and numerical methods; the final saved states are not universal reference answers.